In [ ]:
# (필수) 노트북 위치와 무관하게 src 패키지를 import 할 수 있게 경로를 잡습니다.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project ROOT =", ROOT)


# 08 — Prompt-to-SQL

unsafe vs safe SQL 실행 차이 데모(결정론적).

In [ ]:
import sqlite3, re

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("create table users(id int, name text, role text)")
cur.executemany("insert into users values (?,?,?)", [(1,'kim','user'),(2,'lee','admin'),(3,'park','user')])
conn.commit()

def run_sql_unsafe(sql: str):
    return cur.execute(sql).fetchall()

def run_sql_safe(sql: str):
    s = sql.strip().lower()
    if not s.startswith("select"):
        raise ValueError("Only SELECT is allowed")
    if re.search(r";|\bdrop\b|\bdelete\b|\bupdate\b|\binsert\b|\balter\b", s):
        raise ValueError("Blocked tokens")
    return cur.execute(sql).fetchall()

print("baseline:", run_sql_unsafe("select name, role from users"))
attack1 = "select name, role from users where role='admin' OR 1=1"
print("unsafe attack1:", run_sql_unsafe(attack1))
try:
    print("safe attack1:", run_sql_safe(attack1))
except Exception as e:
    print("safe blocked:", e)

attack2 = "drop table users"
run_sql_unsafe(attack2)
try:
    print(run_sql_unsafe("select * from users"))
except Exception as e:
    print("after drop failed (expected):", e)
